In [1]:
import pandas as pd
import numpy as np
import json

In [2]:
# version_modelo = 'gemma3_1b_it'
version_modelo = 'gemma_sinfinetuning'
# version_modelo = 'llama_v3_beam'

In [3]:
alignscore = pd.read_csv(f'resultados_metricas/alignscore_{version_modelo}.csv', sep=';')
alignscore = alignscore.iloc[list(range(74))+list(range(75,87))+list(range(88,218))]
read_bert = pd.read_csv(f'resultados_metricas/readability_bertscore_{version_modelo}.csv', sep=';')

In [4]:
alignscore.columns

Index(['codigo_comun', 'non_pls', 'pls', 'pls_gemma_sinfinetuning',
       'alignscore'],
      dtype='object')

In [5]:
read_bert.columns

Index(['codigo_comun', 'non_pls', 'pls', 'pls_gemma_sinfinetuning',
       'Coleman-Liau', 'FleschReadingEase', 'GunningFogIndex', 'SMOGIndex',
       'Kincaid', 'DaleChallIndex', 'bertscore_f1'],
      dtype='object')

In [6]:
def compare_and_average(df1: pd.DataFrame, df2: pd.DataFrame, test_name: str, output_filename: str = "results.json") -> tuple[dict | None, pd.DataFrame | None]:
    """
    Compares two DataFrames. If common columns are identical (regardless of
    row order, based on 'codigo_comun'), it calculates the average of the
    non-shared columns, saves the result to a JSON file, and returns the
    results dictionary and a combined DataFrame.

    Metrics in the JSON and columns in the DataFrame are returned in a
    pre-defined order.

    Args:
        df1: The first pandas DataFrame.
        df2: The second pandas DataFrame.
        test_name: A string name for the test run (e.g., 'colab_lora_03_nosample').
        output_filename: The name of the JSON file to save results to.

    Returns:
        A tuple of (results_dict, combined_dataframe):
        - (dict, pd.DataFrame): If successful.
        - (None, None): If the common columns are not identical.
    """
    
    # --- Define the desired metric and column order ---
    METRIC_ORDER = [
        'Coleman-Liau',
        'FleschReadingEase',
        'GunningFogIndex',
        'SMOGIndex',
        'Kincaid',
        'DaleChallIndex',
        'alignscore',
        'bertscore_f1'
    ]
    
    # 1. Identify common and unique columns
    df1_cols = set(df1.columns)
    df2_cols = set(df2.columns)
    
    common_cols = list(df1_cols.intersection(df2_cols))
    df1_unique_cols = list(df1_cols.difference(df2_cols))
    df2_unique_cols = list(df2_cols.difference(df1_cols))
    
    if not common_cols:
        print("Error: No common columns found between the DataFrames.")
        return None, None
        
    print(f"Common columns: {common_cols}")
    print(f"DF1 unique columns: {df1_unique_cols}")
    print(f"DF2 unique columns: {df2_unique_cols}")

    # 2. Check if the data in common columns is identical
    are_equal = False
    if 'codigo_comun' not in common_cols:
        print("\nWarning: 'codigo_comun' not in common columns. Comparing rows as-is.")
        are_equal = df1[common_cols].equals(df2[common_cols])
    else:
        print("\nComparing common columns, sorting by 'codigo_comun' to handle row order...")
        try:
            df1_common_sorted = df1[common_cols].sort_values(by='codigo_comun').reset_index(drop=True)
            df2_common_sorted = df2[common_cols].sort_values(by='codigo_comun').reset_index(drop=True)
            are_equal = df1_common_sorted.equals(df2_common_sorted)
        except Exception as e:
            print(f"Error during sorted comparison: {e}")
            return None, None
    
    if not are_equal:
        print("\nError: Data in common columns is not identical. Aborting.")
        return None, None

    print("\nSuccess: Common columns are identical.")

    # 3. Calculate the average of non-shared columns
    all_averages = {}
    
    # Combine all unique columns to calculate averages
    all_unique_cols = df1_unique_cols + df2_unique_cols
    
    for col in all_unique_cols:
        # Determine which dataframe the column is from
        df_to_use = df1 if col in df1_unique_cols else df2
        
        numeric_col = pd.to_numeric(df_to_use[col], errors='coerce')
        if not numeric_col.isnull().all():
            all_averages[col] = numeric_col.mean()
        else:
            print(f"Warning: Column '{col}' could not be averaged (all non-numeric?).")

    # 4. Prepare the final JSON output structure (in order)
    ordered_averages = {}
    
    # Add metrics in the specified order
    for metric in METRIC_ORDER:
        if metric in all_averages:
            # Use pop() to get value and remove from all_averages
            ordered_averages[metric] = all_averages.pop(metric)
            
    # Add any remaining metrics that were not in METRIC_ORDER
    # (This makes the function robust to unexpected new metrics)
    ordered_averages.update(all_averages)
    
    output_data = {
        "test_name": test_name,
        "averages": ordered_averages
    }

    # 5. Write to the JSON file
    try:
        with open(output_filename, 'w') as f:
            json.dump(output_data, f, indent=4)
        print(f"\nSuccessfully calculated averages and saved to {output_filename}")
    except IOError as e:
        print(f"\nError writing to JSON file {output_filename}: {e}")
        return None, None

    # 6. Create the combined DataFrame
    combined_df = None
    if 'codigo_comun' not in common_cols:
        print("Warning: Cannot create combined DataFrame without 'codigo_comun' as a key.")
        combined_df = pd.concat([df1, df2[df2_unique_cols]], axis=1)
    else:
        df1_indexed = df1.set_index('codigo_comun')
        df2_indexed = df2.set_index('codigo_comun')
        combined_df = df1_indexed.join(df2_indexed[df2_unique_cols])
        combined_df = combined_df.reset_index()

    # 7. Re-order the columns of the combined DataFrame
    current_cols = list(combined_df.columns)
    
    # Define common columns (excluding 'codigo_comun')
    common_order = [col for col in common_cols if col != 'codigo_comun']
    
    # Start with 'codigo_comun'
    final_col_order = ['codigo_comun'] if 'codigo_comun' in current_cols else []
    
    # Add common columns, then ordered metrics
    for col_list in [common_order, METRIC_ORDER]:
        for col in col_list:
            if col in current_cols and col not in final_col_order:
                final_col_order.append(col)
                
    # Add any other columns that weren't in the lists
    for col in current_cols:
        if col not in final_col_order:
            final_col_order.append(col)
            
    combined_df = combined_df[final_col_order]

    return combined_df

In [7]:
res = compare_and_average(alignscore, read_bert, version_modelo, f'resultados_modelos/metricas_{version_modelo}.json')

Common columns: ['pls_gemma_sinfinetuning', 'pls', 'non_pls', 'codigo_comun']
DF1 unique columns: ['alignscore']
DF2 unique columns: ['DaleChallIndex', 'FleschReadingEase', 'bertscore_f1', 'Coleman-Liau', 'Kincaid', 'SMOGIndex', 'GunningFogIndex']

Comparing common columns, sorting by 'codigo_comun' to handle row order...

Success: Common columns are identical.

Successfully calculated averages and saved to resultados_modelos/metricas_gemma_sinfinetuning.json


In [8]:
res.to_csv(f'resultados_metricas/metricas_{version_modelo}.csv', sep=';', index=False)